# 05b Calibrate Watershed

Wflow Readiness: validate the Wflow discharge **response** against observed USGS instantaneous (IV) records for **historical validation events** (design rows whose rainfall analog runs at ~unit scale — real observed storms), scoring simulated-vs-observed hourly hydrographs (KGE/NSE, peak/volume bias). Also reads the single-K Same-Frequency Amplification provenance and the baseflow validation. Discharge is generated from rainfall + antecedent moisture; there is no injected streamflow member.

## Runtime

Load the current location config, catalog, readiness tables, and Wflow/SFINCS paths.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

import geopandas as gpd
import xarray as xr

location_root = Path("..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

from wflow_runs.calibration import (
    active_domain_handoff,
    build_usgs_wflow_calibration,
    cache_validation_iv_records,
    plot_handoff_discharge_audit,
    plot_usgs_observation_comparison,
    plot_wflow_sfincs_gauge_output,
    post_run_handoff_readout,
    run_wflow_validation_audit,
    select_validation_events,
    summarize_handoff_hydrographs,
)
from wflow_runs.notebook import load_runtime

runtime = load_runtime(location_root)
location_name = runtime.location_name
config = runtime.config
paths = runtime.paths
scenario_catalog_path = runtime.scenario_catalog_path
readiness_path = runtime.readiness_path
joint_worklist_path = runtime.joint_worklist_path
events_root = runtime.events_root
wflow_base_root = runtime.wflow_base_root
streamflow_records_path = runtime.streamflow_records_path
event_streamflow_iv_root = runtime.event_streamflow_iv_root
audit_plots_dir = runtime.audit_plots_dir

pd.Series({
    "location": location_name,
    "scenario_catalog": str(scenario_catalog_path),
    "readiness": str(readiness_path),
    "joint_worklist": str(joint_worklist_path),
    "events_root": str(events_root),
}, name="wflow_calibration_runtime")


## Rerun Control


In [ ]:
rerun = False


## Current Scenario Sample

Randomly sample from the current runnable scenario set. `blocked` means pending Wflow handoff execution; `accepted` means a handoff artifact already exists. `incompatible` rows are excluded from the sample and should normally be zero after catalog repair.


In [ ]:
# Validate Wflow against historical-like events so simulated hydrographs can be scored
# against observed USGS instantaneous streamflow.
VALIDATION_EVENT_N = 5

validation = select_validation_events(
    config,
    scenario_catalog_path=scenario_catalog_path,
    readiness_path=readiness_path,
    joint_worklist_path=joint_worklist_path,
    n=VALIDATION_EVENT_N,
)
audit_scenarios = validation.scenarios
WFLOW_AUDIT_EVENT_IDS = validation.event_ids

display(validation.summary)
display(audit_scenarios[[column for column in [
    "event_id", "sample_rp_years", "severity_band", "rainfall_member_id",
    "rainfall_member_time", "rainfall_scale_factor",
] if column in audit_scenarios]])


## Cache USGS IV Validation Records

Fetch observed USGS instantaneous-value hydrographs for the selected historical validation event windows and align them to the Wflow output timestep at the same `gauges_usgs` locations.

In [ ]:
FETCH_USGS_IV_VALIDATION_DATA = True

usgs_iv_cache_report = cache_validation_iv_records(
    config,
    location_root,
    WFLOW_AUDIT_EVENT_IDS,
    scenario_catalog_path=scenario_catalog_path,
    events_root=events_root,
    wflow_base_root=wflow_base_root,
    event_streamflow_iv_root=event_streamflow_iv_root,
    rerun=rerun,
    fetch=FETCH_USGS_IV_VALIDATION_DATA,
)

display(usgs_iv_cache_report)
if not usgs_iv_cache_report.empty and not usgs_iv_cache_report["status"].isin(["cached", "fetched"]).all():
    display(usgs_iv_cache_report[~usgs_iv_cache_report["status"].isin(["cached", "fetched"])])


## Wflow Forcing Completeness

Check that the sampled catalog rows carry the file/time fields Wflow needs to stage event precipitation.


In [ ]:
REQUIRED_WFLOW_FORCING_COLUMNS = ["rainfall_member_file", "rainfall_member_id", "rainfall_member_time"]

missing_forcing_rows = []
for event_id in WFLOW_AUDIT_EVENT_IDS:
    row = catalog.loc[catalog["event_id"].astype(str) == str(event_id)].iloc[0]
    for column in REQUIRED_WFLOW_FORCING_COLUMNS:
        value = row.get(column, pd.NA)
        if pd.isna(value) or str(value).strip() == "":
            missing_forcing_rows.append({"event_id": event_id, "missing_field": column})

forcing_completeness = pd.DataFrame(missing_forcing_rows)
display(
    pd.Series(
        {
            "sampled_events_checked": len(WFLOW_AUDIT_EVENT_IDS),
            "missing_wflow_forcing_values": len(forcing_completeness),
        },
        name="wflow_forcing_completeness",
    )
)
display(forcing_completeness)
if not forcing_completeness.empty:
    raise ValueError("Sampled catalog rows are missing required Wflow forcing fields")


## Rainfall-Driven Readiness

Discharge is the Wflow response (no injected streamflow member). Validate the rainfall-driven readiness contract for the selected historical validation events: a rainfall member is wired and the Same-Frequency Amplification / baseflow Primary Reference Gage is configured.

In [ ]:
# confirms catalog events have the discharge records Wflow expects.
from wflow_runs.streamflow_realization import validate_wflow_streamflow_realization

readiness_rows = []
for event_id in WFLOW_AUDIT_EVENT_IDS:
    report = validate_wflow_streamflow_realization(
        config, location_root, event_id, catalog_path=scenario_catalog_path, raise_on_error=False
    )
    for _, r in report.iterrows():
        readiness_rows.append({"event_id": event_id, **r.to_dict()})
readiness_diagnosis = pd.DataFrame(readiness_rows)
display(readiness_diagnosis)
if not readiness_diagnosis.empty and readiness_diagnosis["status"].eq("failed").any():
    raise RuntimeError("Validation events are missing required rainfall forcing for Wflow generation.")


## Handoff Hydrograph Diagnostics

Summarize existing event and zero-rain SFINCS handoff hydrographs for the random sample and any already completed local Wflow events.


In [ ]:
active_domain = active_domain_handoff(location_root)
expected_handoff_ids = active_domain["expected_handoff_ids"]
ACTIVE_WFLOW_SUBMODEL_ID = active_domain["active_wflow_submodel_id"]

active_domain


In [ ]:
hydrograph_event_ids = list(WFLOW_AUDIT_EVENT_IDS)
hydrograph_review = summarize_handoff_hydrographs(config, location_root, hydrograph_event_ids)
hydrograph_summary = hydrograph_review.summary
hydrograph_pairs = hydrograph_review.pairs

display(pd.Series({"hydrograph_event_ids_checked": hydrograph_event_ids}, name="handoff_hydrograph_inputs"))
display(hydrograph_summary)
display(hydrograph_pairs)


In [ ]:
if hydrograph_summary.empty:
    zero_rain_comparison = pd.DataFrame()
else:
    pivot = hydrograph_summary.pivot_table(index=["event_id", "source_id"], columns="kind", values="peak_m3s", aggfunc="first")
    if {"event", "zero_rain"}.issubset(set(pivot.columns)):
        pivot["zero_over_event_peak_fraction"] = pivot["zero_rain"] / pivot["event"]
    zero_rain_comparison = pivot.reset_index()
zero_rain_comparison


In [ ]:
def plot_handoff_discharge(event_id):
    return plot_handoff_discharge_audit(
        config,
        location_root,
        event_id,
        audit_plots_dir=audit_plots_dir,
        expected_handoff_ids=expected_handoff_ids,
    )


## Restart State And Optional Wflow Audit Run

The sampled Wflow runs use the production dynamic handoff path with the configured warmup/restart state, event meteorology, external streamflow realization, and SFINCS handoff gauges.


In [ ]:
restart_state_path = wflow_base_root / ACTIVE_WFLOW_SUBMODEL_ID / "instate" / "instates.nc"
baseline_root = location_root / config.get("wflow", {}).get("dynamic_handoff", {}).get("baseline_root", "data/wflow/warmup/baseline_90d")
restart_state_readiness = pd.Series(
    {
        "active_wflow_submodel": ACTIVE_WFLOW_SUBMODEL_ID,
        "restart_state_path": str(restart_state_path),
        "restart_state_exists": restart_state_path.exists(),
        "baseline_root": str(baseline_root),
        "baseline_root_exists": baseline_root.exists(),
        "state_policy": config.get("wflow", {}).get("dynamic_handoff", {}).get("state_policy"),
        "warmup_days": config.get("wflow", {}).get("dynamic_handoff", {}).get("warmup_days"),
        "baseline_reference_time": config.get("wflow", {}).get("dynamic_handoff", {}).get("baseline_reference_time"),
    },
    name="restart_state_readiness",
)
restart_state_readiness


In [ ]:
EXECUTE_WFLOW_AUDIT = True

display(pd.Series({"wflow_validation_event_ids": WFLOW_AUDIT_EVENT_IDS}, name="historical_validation_events"))

wflow_validation_audit = run_wflow_validation_audit(
    config,
    location_root,
    WFLOW_AUDIT_EVENT_IDS,
    scenario_catalog_path=scenario_catalog_path,
    rerun=rerun,
    execute=EXECUTE_WFLOW_AUDIT,
)
warmup_audit_plan = wflow_validation_audit.plan
wflow_audit_report = wflow_validation_audit.report

display(warmup_audit_plan)
wflow_audit_report


## Post-Run Handoff Diagnostics

After Wflow execution, inspect event and zero-rain handoff hydrographs for the random sample.


In [ ]:
for event_id in WFLOW_AUDIT_EVENT_IDS:
    plot_handoff_discharge(event_id)


In [ ]:
post_run_readout = post_run_handoff_readout(
    config,
    location_root,
    WFLOW_AUDIT_EVENT_IDS,
    streamflow_records_path=streamflow_records_path,
)
post_run_zero_rain_diagnostics = post_run_readout.zero_rain
amplification_readout = post_run_readout.amplification
baseflow_readout = post_run_readout.baseflow

display(post_run_zero_rain_diagnostics)
display(pd.Series({"single_K_amplification": "per-event provenance"}, name="amplification"))
display(amplification_readout)
display(pd.Series({"baseflow_validation": "zero-rain control vs observed low-flow at Primary Reference Gage"}, name="baseflow"))
display(baseflow_readout)


## Wflow Gauge Output And USGS Comparison

Read Wflow native scalar outputs, map `Q_<index>` columns back to SFINCS and USGS gauge layers, and compare simulated/observed hydrographs where records overlap.


In [ ]:
calibration = build_usgs_wflow_calibration(
    config,
    location_root,
    WFLOW_AUDIT_EVENT_IDS,
    events_root=events_root,
    wflow_base_root=wflow_base_root,
    event_streamflow_iv_root=event_streamflow_iv_root,
    active_wflow_submodel_id=ACTIVE_WFLOW_SUBMODEL_ID,
)
calibration_summary = calibration.summary
wflow_calibration_patch = calibration.patch

display(calibration.artifacts)
calibration_summary


In [ ]:
for event_id in WFLOW_AUDIT_EVENT_IDS:
    plot_wflow_sfincs_gauge_output(
        event_id,
        events_root=events_root,
        wflow_base_root=wflow_base_root,
        audit_plots_dir=audit_plots_dir,
        submodel_id=ACTIVE_WFLOW_SUBMODEL_ID,
    )


In [ ]:
for event_id in WFLOW_AUDIT_EVENT_IDS:
    plot_usgs_observation_comparison(
        event_id,
        events_root=events_root,
        wflow_base_root=wflow_base_root,
        event_streamflow_iv_root=event_streamflow_iv_root,
        audit_plots_dir=audit_plots_dir,
        submodel_id=ACTIVE_WFLOW_SUBMODEL_ID,
    )
